In [ ]:
# %% [markdown]
"""
# Polar Vortex Analysis: Composite GIFs for Z3, Temperature, and Potential Vorticity

This notebook analyzes the polar vortex by comparing two experiments ("nocoupl" and "small perturbation"). 
The analysis is based on variables extracted from NetCDF files including geopotential height (Z3), temperature (T), 
and wind components (U, V). In addition, we compute potential vorticity (PV) using MetPy. 

For each experiment, composite daily maps are generated for:
1. **Geopotential Height (Z3) and Wind Vectors:**  
   At 10 hPa and 50 hPa, contour (or filled) plots of Z3 are produced with overlaid (U, V) vectors to visualize the polar vortex center, shape, and its evolution.
2. **Temperature Distribution:**  
   Filled contour plots of temperature at 10 hPa and 50 hPa to identify the warm/cold structure inside/outside the vortex. SSW events may be indicated by rapid warming.
3. **Potential Vorticity (PV):**  
   PV is computed based on U, V, T, and Z3. Composite maps are generated on the 10 hPa and 50 hPa isobaric surfaces and on the 380K isentropic surface.

The experiments follow the same combination as in the O3 analysis code. In order to avoid repeatedly opening the same file,
the data for each year and month are loaded only once, and the required variables are extracted and processed.
GIF animations are created with the same frame duration (500 ms per frame).
"""

# %% 
import os
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
from matplotlib import rc
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.util import add_cyclic_point
import imageio
from PIL import Image
import metpy.calc as mpcalc
from metpy.units import units

# Set matplotlib font settings for consistency
rc('font', **{'family': 'sans-serif', 'sans-serif': ['Helvetica']})
rc('text', usetex=False)

# %% [markdown]
"""
## Helper Functions
Below we define several helper functions:
- **subset_at_pressure**: Extract a variable at a given pressure level (in hPa).
- **compute_potential_temperature**: Compute potential temperature using T and plev.
- **extract_isentropic_surface**: Interpolate a variable to a target potential temperature (e.g., 380K).
- **compute_pv_field**: Compute the potential vorticity field using MetPy.
"""

# %%
def subset_at_pressure(ds, var_name, level_hPa):
    """
    Extract the specified variable at the pressure level (in hPa).
    Note: In the NC files, plev is in Pa so we convert hPa to Pa.
    """
    target_pressure = level_hPa * 100  # convert hPa to Pa
    # Use nearest neighbor selection
    return ds[var_name].sel(plev=target_pressure, method='nearest')

def compute_potential_temperature(T, plev):
    """
    Compute potential temperature: theta = T * (p0 / p)**(R/cp)
    p0 = 100000 Pa; R/cp ~ 0.286.
    T and plev should be xarray objects with compatible dimensions.
    """
    return T * (100000 / plev)**0.286

def extract_isentropic_surface(ds, var_name, theta_target=380):
    """
    Interpolate the specified variable to the target isentropic surface (theta_target in K).
    This function uses a vectorized 1D interpolation along the 'plev' dimension for each (time, lat, lon) point.
    """
    def interp_to_theta(theta, var, theta_target):
        # Check for NaNs; if present, return nan
        if np.any(np.isnan(theta)) or np.any(np.isnan(var)):
            return np.nan
        return np.interp(theta_target, theta, var)
    
    # Compute potential temperature field
    theta = compute_potential_temperature(ds['T'], ds['plev'])
    # Apply the interpolation along the 'plev' dimension
    var_isentropic = xr.apply_ufunc(
        interp_to_theta,
        theta,
        ds[var_name],
        input_core_dims=[['plev'], ['plev']],
        output_core_dims=[[]],
        vectorize=True,
        kwargs={'theta_target': theta_target},
        dask='parallelized',
        output_dtypes=[ds[var_name].dtype]
    )
    return var_isentropic

def compute_pv_field(ds):
    """
    Compute the potential vorticity (PV) field using MetPy's baroclinic PV calculation.
    Assumes ds contains variables: 'U', 'V', 'T', and coordinate 'plev' (in Pa).
    Returns PV with dimensions (time, plev, lat, lon).
    """
    # Assign proper units
    u = ds['U'] * units('m/s')
    v = ds['V'] * units('m/s')
    T = ds['T'] * units('K')
    p = ds['plev'] * units('Pa')
    
    # Compute potential temperature
    theta = T * (100000 * units('Pa') / p)**0.286
    
    # Compute grid spacing (assumes lon and lat are 1D arrays)
    dx, dy = mpcalc.lat_lon_grid_deltas(ds['lon'].values, ds['lat'].values)
    dx = dx * units('m')
    dy = dy * units('m')
    
    # Compute potential vorticity using MetPy (baroclinic formulation)
    # This function expects p, theta, u, v with matching dimensions and dx, dy of shape (lat, lon)
    pv = mpcalc.potential_vorticity_baroclinic(p, theta, u, v, dx, dy)
    return pv

# %% [markdown]
"""
## Data Loading Functions

We define functions to load experiment data from NetCDF files for the two experiments (nocoupl and small perturbation). 
Each function loads February and March data, subsets the latitude range, and averages over ensemble members if applicable.
"""

# %%
def load_experiment_data(file_pattern, lat_min, lat_max):
    """
    Load data from a given file pattern and subset the latitude range.
    If there is an ensemble member dimension, average over it.
    """
    ds = xr.open_mfdataset(file_pattern, concat_dim='member', combine='nested')
    ds = ds.sel(lat=slice(lat_min, lat_max))
    if 'member' in ds.dims:
        ds = ds.mean(dim='member')
    return ds

def load_year_nocouple_data(year, lat_min=60, lat_max=90):
    """
    Load February and March data for the nocoupl experiment for a given year.
    """
    file_Feb = f'/home/weiji/restart_exam/nocouple_data/{year}/Feb_NOCOUPL/{year}-02/*.nc'
    file_Mar = f'/home/weiji/restart_exam/nocouple_data/{year}/Mar_NOCOUPL/{year}-03/*.nc'
    data_Feb = load_experiment_data(file_Feb, lat_min, lat_max)
    data_Mar = load_experiment_data(file_Mar, lat_min, lat_max)
    return data_Feb, data_Mar

def load_year_small_data(year, lat_min=60, lat_max=90):
    """
    Load February and March data for the small perturbation experiment for a given year.
    """
    base_path = f'/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.002_{year}'
    file_Feb = os.path.join(base_path, 'Feb', f'BWCN.e122.f19_g16.002.{year}-02.*.nc')
    file_Mar = os.path.join(base_path, 'Mar', f'BWCN.e122.f19_g16.002.{year}-03.*.nc')
    data_Feb = load_experiment_data(file_Feb, lat_min, lat_max)
    data_Mar = load_experiment_data(file_Mar, lat_min, lat_max)
    return data_Feb, data_Mar

# %% [markdown]
"""
## Plotting Functions for Composite Maps

For each analysis (Z3 with wind vectors, Temperature, and PV), we define a function that:
- Loops over time frames (days),
- Computes the anomaly with respect to a reference field,
- Plots the two experiments side by side in polar projection,
- Saves each frame as a PNG.

Afterward, these frames are combined into a GIF.
"""

# %%
def plot_composite_maps_z3_wind(exp1_name, data1, exp2_name, data2, ref_field, output_dir, level_hPa, frame_duration=500):
    """
    Plot composite daily maps for Z3 (geopotential height) with overlaid U,V wind vectors.
    The variable anomaly is computed as (experiment - reference).
    Data are extracted at the specified pressure level (in hPa).
    """
    os.makedirs(output_dir, exist_ok=True)
    
    # Set fixed color limits (example values, adjust as needed)
    if level_hPa == 10:
        vmin, vmax = -200, 200  # in meters
    elif level_hPa == 50:
        vmin, vmax = -100, 100
    else:
        vmin, vmax = None, None

    n_frames = min(data1.time.size, data2.time.size)
    
    for i in range(n_frames):
        # Extract time slice for both experiments
        ds1 = data1.isel(time=i)
        ds2 = data2.isel(time=i)
        
        # Extract Z3 at the given pressure level
        z3_1 = subset_at_pressure(ds1, 'Z3', level_hPa)
        z3_2 = subset_at_pressure(ds2, 'Z3', level_hPa)
        ref_z3 = subset_at_pressure(ref_field, 'Z3', level_hPa)
        
        # Compute anomaly (experiment - reference)
        diff1 = z3_1 - ref_z3
        diff2 = z3_2 - ref_z3
        
        # Extract wind components at the same level
        u1 = subset_at_pressure(ds1, 'U', level_hPa)
        v1 = subset_at_pressure(ds1, 'V', level_hPa)
        u2 = subset_at_pressure(ds2, 'U', level_hPa)
        v2 = subset_at_pressure(ds2, 'V', level_hPa)
        
        # Add cyclic point for proper mapping (if lon coordinate exists)
        if 'lon' in diff1.coords and diff1.lon.size > 0:
            diff1_vals, lons = add_cyclic_point(diff1.values, coord=diff1.lon.values)
            diff2_vals, _ = add_cyclic_point(diff2.values, coord=diff2.lon.values)
            u1_vals, _ = add_cyclic_point(u1.values, coord=u1.lon.values)
            v1_vals, _ = add_cyclic_point(v1.values, coord=v1.lon.values)
            u2_vals, _ = add_cyclic_point(u2.values, coord=u2.lon.values)
            v2_vals, _ = add_cyclic_point(v2.values, coord=v2.lon.values)
        else:
            print(f"Frame {i+1}: 'lon' coordinate missing, skipping frame.")
            continue
        
        # Compute area-average anomalies for display
        avg1 = float(diff1.mean(dim=['lat', 'lon']).values)
        avg2 = float(diff2.mean(dim=['lat', 'lon']).values)
        
        # Get time label
        time_val = np.datetime64(ds1.time.values)
        time_label = np.datetime_as_string(time_val, unit='D')
        
        # Create figure with two polar subplots
        fig, axes = plt.subplots(1, 2, subplot_kw={'projection': ccrs.NorthPolarStereo()}, figsize=(16,8))
        for ax in axes:
            ax.set_extent([-180, 180, 60, 90], ccrs.PlateCarree())
            ax.coastlines()
            ax.add_feature(cfeature.LAND, facecolor='lightgray')
            gl = ax.gridlines(draw_labels=True, linestyle='--', color='gray')
            gl.xlabels_top = False
            gl.ylabels_right = False
        
        # Left panel: nocoupl experiment
        cf1 = axes[0].contourf(lons, diff1.lat, diff1_vals, levels=np.linspace(vmin, vmax, 21),
                                 cmap='RdBu_r', extend='both', transform=ccrs.PlateCarree())
        axes[0].quiver(lons[::5], diff1.lat.values[::5], u1_vals[::5, ::5], v1_vals[::5, ::5],
                       transform=ccrs.PlateCarree(), scale=500)
        axes[0].set_title(f"{exp1_name} (Anomaly Avg: {avg1:.2f} m)\nDate: {time_label}", fontsize=14)
        
        # Right panel: small perturbation experiment
        cf2 = axes[1].contourf(lons, diff2.lat, diff2_vals, levels=np.linspace(vmin, vmax, 21),
                                 cmap='RdBu_r', extend='both', transform=ccrs.PlateCarree())
        axes[1].quiver(lons[::5], diff2.lat.values[::5], u2_vals[::5, ::5], v2_vals[::5, ::5],
                       transform=ccrs.PlateCarree(), scale=500)
        axes[1].set_title(f"{exp2_name} (Anomaly Avg: {avg2:.2f} m)\nDate: {time_label}", fontsize=14)
        
        # Add common colorbar
        cbar = fig.colorbar(cf2, ax=axes, orientation='horizontal', fraction=0.05, pad=0.08)
        cbar.set_label(f"Z3 Anomaly (m) at {level_hPa} hPa", fontsize=12)
        
        fig.suptitle(f"Composite Z3 Anomaly (Experiment - Reference)\nPressure Level: {level_hPa} hPa", fontsize=16)
        
        out_fname = os.path.join(output_dir, f"frame_{i+1:03d}.png")
        plt.savefig(out_fname, dpi=150, bbox_inches='tight')
        plt.close(fig)
        print(f"Saved frame {i+1} to {out_fname}")

def plot_composite_maps_temperature(exp1_name, data1, exp2_name, data2, ref_field, output_dir, level_hPa, frame_duration=500):
    """
    Plot composite daily maps for Temperature anomalies at the given pressure level (in hPa).
    """
    os.makedirs(output_dir, exist_ok=True)
    
    # Set fixed color limits (example values)
    if level_hPa == 10:
        vmin, vmax = -30, 30  # in K
    elif level_hPa == 50:
        vmin, vmax = -20, 20
    else:
        vmin, vmax = None, None
    
    n_frames = min(data1.time.size, data2.time.size)
    
    for i in range(n_frames):
        ds1 = data1.isel(time=i)
        ds2 = data2.isel(time=i)
        
        # Extract Temperature at the specified level
        t1 = subset_at_pressure(ds1, 'T', level_hPa)
        t2 = subset_at_pressure(ds2, 'T', level_hPa)
        ref_t = subset_at_pressure(ref_field, 'T', level_hPa)
        
        diff1 = t1 - ref_t
        diff2 = t2 - ref_t
        
        if 'lon' in diff1.coords and diff1.lon.size > 0:
            diff1_vals, lons = add_cyclic_point(diff1.values, coord=diff1.lon.values)
            diff2_vals, _ = add_cyclic_point(diff2.values, coord=diff2.lon.values)
        else:
            print(f"Frame {i+1}: 'lon' coordinate missing, skipping frame.")
            continue
        
        avg1 = float(diff1.mean(dim=['lat','lon']).values)
        avg2 = float(diff2.mean(dim=['lat','lon']).values)
        
        time_val = np.datetime64(ds1.time.values)
        time_label = np.datetime_as_string(time_val, unit='D')
        
        fig, axes = plt.subplots(1, 2, subplot_kw={'projection': ccrs.NorthPolarStereo()}, figsize=(16,8))
        for ax in axes:
            ax.set_extent([-180, 180, 60, 90], ccrs.PlateCarree())
            ax.coastlines()
            ax.add_feature(cfeature.LAND, facecolor='lightgray')
            gl = ax.gridlines(draw_labels=True, linestyle='--', color='gray')
            gl.xlabels_top = False
            gl.ylabels_right = False
        
        cf1 = axes[0].contourf(lons, diff1.lat, diff1_vals, levels=np.linspace(vmin, vmax, 21),
                                 cmap='coolwarm', extend='both', transform=ccrs.PlateCarree())
        axes[0].set_title(f"{exp1_name} (Anomaly Avg: {avg1:.2f} K)\nDate: {time_label}", fontsize=14)
        
        cf2 = axes[1].contourf(lons, diff2.lat, diff2_vals, levels=np.linspace(vmin, vmax, 21),
                                 cmap='coolwarm', extend='both', transform=ccrs.PlateCarree())
        axes[1].set_title(f"{exp2_name} (Anomaly Avg: {avg2:.2f} K)\nDate: {time_label}", fontsize=14)
        
        cbar = fig.colorbar(cf2, ax=axes, orientation='horizontal', fraction=0.05, pad=0.08)
        cbar.set_label(f"Temperature Anomaly (K) at {level_hPa} hPa", fontsize=12)
        
        fig.suptitle(f"Composite Temperature Anomaly (Experiment - Reference)\nPressure Level: {level_hPa} hPa", fontsize=16)
        
        out_fname = os.path.join(output_dir, f"frame_{i+1:03d}.png")
        plt.savefig(out_fname, dpi=150, bbox_inches='tight')
        plt.close(fig)
        print(f"Saved frame {i+1} to {out_fname}")

def plot_composite_maps_pv(exp1_name, data1, data2, ref_field, output_dir, target, frame_duration=500):
    """
    Plot composite daily maps for Potential Vorticity (PV) anomalies.
    The target can be a pressure level in hPa (e.g., 10 or 50) or a string '380K' for the isentropic surface.
    """
    os.makedirs(output_dir, exist_ok=True)
    
    # Compute PV field for each dataset using the full vertical profile
    pv1_full = compute_pv_field(data1)
    pv2_full = compute_pv_field(data2)
    pv_ref_full = compute_pv_field(ref_field)
    
    # Determine slice based on target
    if isinstance(target, (int, float)):
        # Target is a pressure level in hPa -> convert to Pa
        target_Pa = target * 100
        pv1 = pv1_full.sel(plev=target_Pa, method='nearest')
        pv2 = pv2_full.sel(plev=target_Pa, method='nearest')
        pv_ref = pv_ref_full.sel(plev=target_Pa, method='nearest')
        title_target = f"{target} hPa"
    elif isinstance(target, str) and target.endswith("K"):
        # Target is an isentropic surface
        theta_target = float(target[:-1])
        pv1 = extract_isentropic_surface(data1, 'T', theta_target)  # As an example, using T to represent PV surface
        pv2 = extract_isentropic_surface(data2, 'T', theta_target)
        pv_ref = extract_isentropic_surface(ref_field, 'T', theta_target)
        title_target = f"{target} isentropic surface"
        # Note: For proper PV on isentropic surfaces, one should compute PV by interpolating the full 3D PV field.
        # Here we use a simplified approach for demonstration.
    else:
        raise ValueError("Target for PV must be a pressure level (number) or a string ending with 'K'.")
    
    # Set color limits (example values, adjust as needed)
    vmin, vmax = -3e-6, 3e-6  # in PVU (1 PVU = 1e-6 K m^2/kg/s)
    
    n_frames = min(data1.time.size, data2.time.size)
    
    for i in range(n_frames):
        # For each time frame, extract PV slice
        if isinstance(target, (int, float)):
            pv1_frame = pv1.isel(time=i)
            pv2_frame = pv2.isel(time=i)
            pv_ref_frame = pv_ref.isel(time=i)
        else:
            # For isentropic, pv1, pv2, pv_ref are already 2D fields
            pv1_frame = pv1.isel(time=i)
            pv2_frame = pv2.isel(time=i)
            pv_ref_frame = pv_ref.isel(time=i)
        
        diff1 = pv1_frame - pv_ref_frame
        diff2 = pv2_frame - pv_ref_frame
        
        if 'lon' in diff1.coords and diff1.lon.size > 0:
            diff1_vals, lons = add_cyclic_point(diff1.values, coord=diff1.lon.values)
            diff2_vals, _ = add_cyclic_point(diff2.values, coord=diff2.lon.values)
        else:
            print(f"Frame {i+1}: 'lon' coordinate missing, skipping frame.")
            continue
        
        avg1 = float(diff1.mean(dim=['lat','lon']).values)
        avg2 = float(diff2.mean(dim=['lat','lon']).values)
        
        time_val = np.datetime64(data1.time.values[i])
        time_label = np.datetime_as_string(time_val, unit='D')
        
        fig, axes = plt.subplots(1, 2, subplot_kw={'projection': ccrs.NorthPolarStereo()}, figsize=(16,8))
        for ax in axes:
            ax.set_extent([-180, 180, 60, 90], ccrs.PlateCarree())
            ax.coastlines()
            ax.add_feature(cfeature.LAND, facecolor='lightgray')
            gl = ax.gridlines(draw_labels=True, linestyle='--', color='gray')
            gl.xlabels_top = False
            gl.ylabels_right = False
        
        cf1 = axes[0].contourf(lons, diff1.lat, diff1_vals, levels=np.linspace(vmin, vmax, 21),
                                 cmap='BrBG', extend='both', transform=ccrs.PlateCarree())
        axes[0].set_title(f"{exp1_name} (Anomaly Avg: {avg1:.2e} PVU)\nDate: {time_label}", fontsize=14)
        
        cf2 = axes[1].contourf(lons, diff2.lat, diff2_vals, levels=np.linspace(vmin, vmax, 21),
                                 cmap='BrBG', extend='both', transform=ccrs.PlateCarree())
        axes[1].set_title(f"{exp2_name} (Anomaly Avg: {avg2:.2e} PVU)\nDate: {time_label}", fontsize=14)
        
        cbar = fig.colorbar(cf2, ax=axes, orientation='horizontal', fraction=0.05, pad=0.08)
        cbar.set_label(f"PV Anomaly (PVU) at {title_target}", fontsize=12)
        
        fig.suptitle(f"Composite PV Anomaly (Experiment - Reference)\nSurface: {title_target}", fontsize=16)
        
        out_fname = os.path.join(output_dir, f"frame_{i+1:03d}.png")
        plt.savefig(out_fname, dpi=150, bbox_inches='tight')
        plt.close(fig)
        print(f"Saved frame {i+1} to {out_fname}")

# %% [markdown]
"""
## Utility Function to Create GIF from PNG Frames
"""

# %%
def create_gif_from_directory(image_dir, output_gif, duration=500):
    """
    Combine PNG images from a directory into a GIF.
    """
    files = sorted([f for f in os.listdir(image_dir) if f.endswith('.png')])
    if not files:
        print(f"No PNG images found in {image_dir}!")
        return

    images = []
    for fname in files:
        fpath = os.path.join(image_dir, fname)
        try:
            images.append(imageio.imread(fpath))
        except Exception as e:
            print(f"Error reading {fpath}: {e}")
    imageio.mimsave(output_gif, images, duration=duration/1000)
    print(f"Saved GIF: {output_gif}")

# %% [markdown]
"""
## Main Program

We loop over the defined experiment years and months. A reference dataset (covering January–May) is loaded once 
to compute 2D time-averaged reference fields for each variable. Then, for each combination of year, month, and target level,
composite maps for Z3 (with wind vectors), Temperature, and PV are generated and saved as PNG frames which are then combined into GIFs.
"""

# %%
# Base output directory for composite analysis
base_output_dir = "/home/weiji/restart_exam/20250221/polar_vortex_analysis_G"
os.makedirs(base_output_dir, exist_ok=True)

# Define experiment years and months (same as O3 analysis)
years = ['0008', '0013', '0014', '0019']
months = ['Feb', 'Mar']

# Load reference dataset once (contains all years)
ref_path = "/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.002/BWCN.e122.f19_g16.002.cam.h3.0001-0023.int.nc"
ds_ref_all = xr.open_dataset(ref_path)
ds_ref_all = ds_ref_all.sel(lat=slice(60, 90))

# Loop over each year
for year in years:
    print(f"\nProcessing Year: {year}")
    # Select reference data for current year and months January–May
    ref_sel = ds_ref_all.sel(time=(ds_ref_all.time.dt.year == int(year)) & 
                             (ds_ref_all.time.dt.month.isin([1,2,3,4,5]))).sortby('time')
    # For each variable, compute time-averaged 2D reference field
    ref_z3 = ref_sel['Z3'].mean(dim='time')
    ref_T = ref_sel['T'].mean(dim='time')
    # For PV, compute from the full reference dataset
    ref_ds = ref_sel  # use the selected reference dataset
    
    for month in months:
        print(f"Processing Month: {month}")
        # Load experimental data for both experiments
        if month == 'Feb':
            noc_ds_Feb, _ = load_year_nocouple_data(year)
            small_ds_Feb, _ = load_year_small_data(year)
            data_nocouple = noc_ds_Feb
            data_small = small_ds_Feb
        elif month == 'Mar':
            _, noc_ds_Mar = load_year_nocouple_data(year)
            _, small_ds_Mar = load_year_small_data(year)
            data_nocouple = noc_ds_Mar
            data_small = small_ds_Mar
        else:
            continue
        
        # ------------------------------
        # 1. Z3 with Wind Vectors at 10 and 50 hPa
        # ------------------------------
        for level in [10, 50]:
            combo_dir = os.path.join(base_output_dir, f"{year}_{month}_Z3_{level}hPa")
            os.makedirs(combo_dir, exist_ok=True)
            plot_composite_maps_z3_wind("nocoupl", data_nocouple, "small", data_small,
                                        ref_sel, combo_dir, level, frame_duration=500)
            gif_fname = os.path.join(base_output_dir, f"composite_{year}_{month}_Z3_{level}hPa.gif")
            create_gif_from_directory(combo_dir, gif_fname, duration=500)
        
        # ------------------------------
        # 2. Temperature at 10 and 50 hPa
        # ------------------------------
        for level in [10, 50]:
            combo_dir = os.path.join(base_output_dir, f"{year}_{month}_T_{level}hPa")
            os.makedirs(combo_dir, exist_ok=True)
            plot_composite_maps_temperature("nocoupl", data_nocouple, "small", data_small,
                                            ref_sel, combo_dir, level, frame_duration=500)
            gif_fname = os.path.join(base_output_dir, f"composite_{year}_{month}_T_{level}hPa.gif")
            create_gif_from_directory(combo_dir, gif_fname, duration=500)
        
        # ------------------------------
        # 3. Potential Vorticity (PV) at 10 hPa, 50 hPa, and 380K isentropic surface
        # ------------------------------
        for target in [10, 50, "380K"]:
            combo_dir = os.path.join(base_output_dir, f"{year}_{month}_PV_{target}")
            os.makedirs(combo_dir, exist_ok=True)
            plot_composite_maps_pv("nocoupl", data_nocouple, data_small, ref_sel,
                                   combo_dir, target, frame_duration=500)
            gif_fname = os.path.join(base_output_dir, f"composite_{year}_{month}_PV_{target}.gif")
            create_gif_from_directory(combo_dir, gif_fname, duration=500)

print("\nAll composite GIFs have been generated!")
